In [ ]:
import mlflow
import pandas as pd
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier
import plotly.graph_objects as go
import numpy as np

# ============ CONFIGURATION ============
window = 7
boruta_max_iter = 100
boruta_alpha = 0.05
boruta_perc = 100
rf_max_depth = 5
boruta_n_estimators = "auto"
# =======================================

# ============ LOAD LABELLED DATA ============
labelled_data_cache = pd.read_pickle('data_cache/04_labelled_data.pkl')
X_train = labelled_data_cache['X_train']
y_train = labelled_data_cache['y_train']
profit_target = labelled_data_cache['profit_target']
stop_loss = labelled_data_cache['stop_loss']

In [ ]:
# =========== MLFLOW SETUP ============
mlflow.set_experiment("05_feature_selection")

# ============ TRAIN MODEL ============
with mlflow.start_run(run_name="boruta_method_features"):
    # Boruta Parameters
    mlflow.log_params({
        "method": "BorutaPy",
        "estimator": "RandomForestClassifier",
        "n_estimators": "auto",
        "max_iter": boruta_max_iter,
        "alpha": boruta_alpha,
        "perc": boruta_perc,
        "two_step": True,
        "target": f"Label_{window}day",
        "n_input_features": X_train.shape[1],
        "train_size": len(X_train),
    })

    borutaRandomForest = RandomForestClassifier(n_jobs=-1, class_weight="balanced",
                                max_depth=rf_max_depth, random_state=42)
    
    featureSelector = BorutaPy(borutaRandomForest, n_estimators=boruta_n_estimators, max_iter=boruta_max_iter,
                        alpha=boruta_alpha, perc=boruta_perc, verbose=2, random_state=42)
    
    featureSelector.fit(X_train.values, y_train.values)

    confirmedFeatures  = X_train.columns[featureSelector.support_].tolist() #The mask of selected features - only confirmed ones are True
    tentativeFeatures  = X_train.columns[featureSelector.support_weak_].tolist()
    rejectedFeatures   = X_train.columns[~featureSelector.support_ & ~featureSelector.support_weak_].tolist()
    
    # Log X/y_train
    fs_labelled_data = X_train.join(y_train).copy()
    mlflow.log_input(
        mlflow.data.from_pandas(
            fs_labelled_data,  # single DataFrame with features + labels
            name="fs_labelled_data",
            targets=y_train.name    # tells MLflow which column is the label
        ),
        context="feature selection"
    )
    fs_labelled_data.to_pickle('data_cache/05_fs_labelled_data.pkl')
    mlflow.log_artifact('data_cache/05_fs_labelled_data.pkl')
    
    # Log metrics
    mlflow.log_metrics({
        "n_confirmed": len(confirmedFeatures),
        "n_tentative": len(tentativeFeatures),
        "n_rejected":  len(rejectedFeatures),
    })
    
    # Log triple barrier parameters
    mlflow.log_param("profit_target", profit_target)
    mlflow.log_param("stop_loss", stop_loss)
    
    # Log features as parameters
    for feature in confirmedFeatures:
        mlflow.log_param(f"status_{feature}", "Confirmed")
    for feature in tentativeFeatures:
        mlflow.log_param(f"status_{feature}", "Tentative")
    for feature in rejectedFeatures:
        mlflow.log_param(f"status_{feature}", "Rejected")
    
    # Log features dataframe as artifact
    feature_status = pd.DataFrame({
        "feature": confirmedFeatures + tentativeFeatures + rejectedFeatures,
        "status":  (["confirmed"] * len(confirmedFeatures) +
                    ["tentative"] * len(tentativeFeatures) +
                    ["rejected"]  * len(rejectedFeatures))
    })
    feature_status.to_pickle("data_cache/05_feature_status.pkl")
    mlflow.log_artifact("data_cache/05_feature_status.pkl")
        
    features     = np.array(X_train.columns.tolist())
    colour_map   = {"confirmed": "#2ecc71", "tentative": "#f39c12", "rejected": "#e74c3c"}
    
    history     = featureSelector.importance_history_
    status_map = feature_status.set_index("feature")["status"].to_dict()
    colours     = [colour_map[status_map[f]] for f in features]

    W, H = int((18/2.54)*150), int((9/2.54)*150)

    fig_box = go.Figure([go.Box(y=history[:, i], name=f, marker_color=colours[i], showlegend=False)
                        for i, f in enumerate(features)])
    
    fig_box.update_layout(
        title="Boruta Importance History",
        xaxis=dict(tickmode="linear", tickangle=-45, tickfont=dict(size=10)),
        margin=dict(b=120)
    )
    for label, colour in colour_map.items():
        fig_box.add_trace(go.Box(visible="legendonly", name=label.title(), marker_color=colour))
        
    fig_box.write_image("data_cache/boruta_importance_history.png", width=W, height=H)
    mlflow.log_artifact("data_cache/boruta_importance_history.png")